# Art Director Art Director

You play the role of an **art director**: you give a short creative intention in plain language, and the system turns it into a finished image. A local language model (LM Studio) expands your intention once into a detailed visual prompt, and the same prompt is then compared with Stable Diffusion 1.5 and SD-Turbo. The diffusion runtime is deliberately loaded only after the prompt step succeeds.

**🎯 What you will learn**

- The difference between a user *intention* and a *visual prompt*, and what an LLM adds (subject, style, lighting, composition, mood).
- How diffusion models turn noise into an image, step by step.
- Multi-step Stable Diffusion 1.5 vs. one-step SD-Turbo, and the speed/quality trade-off.
- How `steps`, `guidance_scale`, negative prompts, and `seed` shape a result, and how to compare two models fairly.

![Art Director Flow](images/art_director_flow.png)

## Prerequisites

- An AMD GPU/APU listed in AMD's [official ROCm compatibility matrix](https://rocm.docs.amd.com/projects/radeon-ryzen/en/latest/docs/compatibility/compatibility.html). Select the architecture-specific ROCm/PyTorch wheels matching that device in the Docker setup in the project README; this notebook does not assume a particular GFX identifier.
- The shared repository-root `.venv`, created by the Docker setup in the project README, with ROCm/PyTorch wheels matching the active GPU/APU and application requirements installed.
- Download the standard local snapshots with the first notebook code cell from the repository root.
- Start LM Studio, load a chat model, and enable its OpenAI-compatible server at http://127.0.0.1:1234.
- Activate `.venv` with `.venv\Scripts\activate` in a Windows terminal or `source .venv/bin/activate` on Linux/macOS. Enter `Art_Director/`, run `jupyter notebook`, and select the standard **Python 3 (ipykernel)** kernel.
- The initial prompt request has no fallback: an unavailable endpoint, timeout, invalid response, or empty content stops the workflow before a diffusion model is loaded.

The standard workflow does not require a GFX environment variable. For advanced troubleshooting only, compare the active device report with the official compatibility matrix and set `AUP_EXPECTED_GFX` only when an explicit diagnostic expectation is needed. The local diffusion smoke driver is `python tests/smoke_art_director.py --output-dir <output-directory>` from the repository root; it does not emulate LM Studio.

## Step 1: Download Models, Import the Runtime, and Configure Prompt Expansion

This self-contained setup cell downloads the two immutable Hugging Face revisions into `runtime_assets/models/`, verifies the files needed by Diffusers, defines the ROCm-only runtime, and configures LM Studio. Existing snapshots are reused by the Hugging Face cache logic.


In [ ]:
import math
import os
from pathlib import Path
from time import perf_counter
import requests
from IPython.display import display

project_root = Path.cwd().resolve()
import gc
from dataclasses import dataclass
from huggingface_hub import snapshot_download

@dataclass(frozen=True)
class DiffusionModelSpec:
    name: str
    repo_id: str
    revision: str
    local_dir: Path
    default_steps: int
    default_guidance: float
    supports_negative_prompt: bool

MODEL_SPECS = {
    "sd15": DiffusionModelSpec("sd15", "stable-diffusion-v1-5/stable-diffusion-v1-5", "451f4fe16113bff5a5d2269ed5ad43b0592e9a14", Path("runtime_assets/models/sd15"), 20, 7.5, True),
    "sd-turbo": DiffusionModelSpec("sd-turbo", "stabilityai/sd-turbo", "b261bac6fd2cf515557d5d0707481eafa0485ec2", Path("runtime_assets/models/sd-turbo"), 1, 0.0, False),
}
MODEL_ALLOW_PATTERNS = ["model_index.json", "text_encoder/config.json", "unet/config.json", "vae/config.json", "scheduler/scheduler_config.json", "tokenizer/*", "feature_extractor/*", "safety_checker/config.json", "**/*.fp16.safetensors"]

def download_model(spec: DiffusionModelSpec) -> Path:
    spec.local_dir.mkdir(parents=True, exist_ok=True)
    snapshot_download(repo_id=spec.repo_id, revision=spec.revision, local_dir=spec.local_dir, allow_patterns=MODEL_ALLOW_PATTERNS)
    required = ["model_index.json", "text_encoder/model.fp16.safetensors", "unet/diffusion_pytorch_model.fp16.safetensors", "vae/diffusion_pytorch_model.fp16.safetensors"]
    missing = [name for name in required if not (spec.local_dir / name).is_file()]
    if missing: raise RuntimeError(f"Incomplete {spec.name} model snapshot: {missing}")
    return spec.local_dir

for model_spec in MODEL_SPECS.values(): download_model(model_spec)
print("Pinned diffusion model snapshots are ready under runtime_assets/models.")

def require_rocm(expected_gfx=None):
    import re, torch
    if not torch.cuda.is_available() or not getattr(torch.version, "hip", None): raise RuntimeError("A PyTorch ROCm GPU is required; CPU fallback is not supported")
    props = torch.cuda.get_device_properties(0)
    match = re.search(r"gfx[0-9a-f]+", str(getattr(props, "gcnArchName", "")).lower()); gfx = match.group(0) if match else "unknown"
    if expected_gfx and gfx != expected_gfx: raise RuntimeError(f"ROCm device mismatch: expected {expected_gfx}, found {gfx}")

class DiffusionRuntime:
    def __init__(self, spec, expected_gfx=None):
        require_rocm(expected_gfx)
        import torch
        from diffusers import AutoPipelineForText2Image
        self.spec, self._closed = spec, False
        self._pipeline = AutoPipelineForText2Image.from_pretrained(str(spec.local_dir), torch_dtype=torch.float16, variant="fp16", use_safetensors=True, local_files_only=True).to("cuda:0")
        self._pipeline.enable_attention_slicing(); self._pipeline.vae.enable_slicing()
    @property
    def closed(self): return self._closed
    def generate(self, prompt, *, negative_prompt="", height=512, width=512, steps=None, guidance_scale=None, count=1, seed=None):
        if self._closed: raise RuntimeError(f"{self.spec.name} DiffusionRuntime is closed")
        if not isinstance(prompt, str) or not prompt.strip(): raise ValueError("prompt must be non-empty")
        steps = self.spec.default_steps if steps is None else steps; guidance_scale = self.spec.default_guidance if guidance_scale is None else guidance_scale
        if self.spec.name == "sd-turbo" and (steps != 1 or guidance_scale != 0.0 or negative_prompt): raise ValueError("sd-turbo requires one step, guidance 0.0, and no negative prompt")
        import torch
        kwargs=dict(prompt=prompt,height=height,width=width,num_inference_steps=steps,guidance_scale=guidance_scale,num_images_per_prompt=count)
        if self.spec.supports_negative_prompt: kwargs["negative_prompt"] = negative_prompt
        if seed is not None: kwargs["generator"] = torch.Generator(device="cuda:0").manual_seed(seed)
        return self._pipeline(**kwargs).images
    def close(self):
        if self._closed: return
        self._closed=True; self._pipeline=None; gc.collect()
        import torch
        torch.cuda.empty_cache()

EXPECTED_GFX = os.environ.get("AUP_EXPECTED_GFX")
LM_STUDIO_URL = os.environ.get("LM_STUDIO_URL", "http://127.0.0.1:1234/v1/chat/completions")
LM_STUDIO_MODEL = os.environ.get("LM_STUDIO_MODEL", "local-model")
LM_STUDIO_TIMEOUT_SECONDS = os.environ.get("LM_STUDIO_TIMEOUT_SECONDS", "30")
print(f"Art Director project root: {project_root}")
print(f"LM Studio endpoint: {LM_STUDIO_URL}")


## Step 2: Understand the teaching workflow

**🎯 One prompt, two generators, fair comparison.** The user intent is expanded exactly once in Step 6. Steps 7 and 8 reuse the same expanded prompt *and* the same seed, so any visible difference comes from the models themselves—multi-step vs. one-step—rather than from a different prompt or a different random start.

**💡 One model at a time.** Each runtime is owned by the code that creates it and is closed in a `finally` block before the next model is loaded, so SD15 and SD-Turbo never occupy GPU memory at the same time.

## Step 3: Fail-closed LM Studio prompt expansion

The request helper distinguishes endpoint availability, timeout, HTTP failure, invalid JSON/schema, and empty content. It raises an actionable error for each case; it never silently substitutes the original intent when expansion was requested.

**💡 What the LLM adds.** A short intention like "a bird" is under-specified for an image model. The LLM expands it into a visual prompt with concrete subject, style, lighting, composition, and mood—the kind of direction an art director gives an illustrator.

**⚠️ Why fail-closed.** If expansion was requested but the service is unreachable, the workflow stops instead of quietly generating from the bare intent, so you always know which prompt produced an image.

In [ ]:
class PromptExpansionError(RuntimeError):
    """Raised when LM Studio cannot provide usable prompt text."""


def _normalize_lm_content(content: str) -> str:
    normalized_content = content.strip()
    while (
        len(normalized_content) >= 2
        and normalized_content[0] == normalized_content[-1]
        and normalized_content[0] in {'"', "'"}
    ):
        normalized_content = normalized_content[1:-1].strip()
    if not any(
        character not in {'"', "'"} and not character.isspace()
        for character in normalized_content
    ):
        raise PromptExpansionError(
            "LM Studio returned empty or quote-only prompt content after quote and whitespace normalization. "
            "Check the loaded model, token limit, and server logs; no original-prompt fallback is provided."
        )
    return normalized_content

def _parse_lm_studio_timeout_seconds(raw_value: object) -> float:
    """Parse the timeout setting only when an LM Studio request is made."""
    setting_name = "LM_STUDIO_TIMEOUT_SECONDS"
    example = "Set LM_STUDIO_TIMEOUT_SECONDS to a finite positive number such as '30'."
    if isinstance(raw_value, bool):
        raise PromptExpansionError(
            f"{setting_name} must be a finite positive number, not a boolean-like value. {example}"
        )
    if not isinstance(raw_value, str):
        raise PromptExpansionError(
            f"{setting_name} must be a string containing a finite positive number. {example}"
        )

    text = raw_value.strip()
    if not text or text.casefold() in {"true", "false", "yes", "no", "on", "off", "t", "f", "y", "n"}:
        raise PromptExpansionError(
            f"{setting_name} must be a finite positive number; received {raw_value!r}. {example}"
        )
    try:
        timeout_seconds = float(text)
    except (TypeError, ValueError) as exc:
        raise PromptExpansionError(
            f"{setting_name} must be a finite positive number; received {raw_value!r}. {example}"
        ) from exc
    if not math.isfinite(timeout_seconds) or timeout_seconds <= 0:
        raise PromptExpansionError(
            f"{setting_name} must be a finite positive number; received {raw_value!r}. {example}"
        )
    return timeout_seconds


def _request_lm_studio(
    system_prompt: str,
    user_prompt: str,
    *,
    temperature: float,
    max_tokens: int,
) -> str:
    timeout_seconds = _parse_lm_studio_timeout_seconds(LM_STUDIO_TIMEOUT_SECONDS)
    payload = {
        "model": LM_STUDIO_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    try:
        response = requests.post(
            LM_STUDIO_URL,
            json=payload,
            headers={"Content-Type": "application/json"},
            timeout=timeout_seconds,
        )
    except requests.exceptions.Timeout as exc:
        raise PromptExpansionError(
            f"LM Studio timed out after {timeout_seconds:g}s at {LM_STUDIO_URL}. "
            "Start the server, load the configured model, or increase LM_STUDIO_TIMEOUT_SECONDS."
        ) from exc
    except requests.exceptions.ConnectionError as exc:
        raise PromptExpansionError(
            f"LM Studio is unavailable at {LM_STUDIO_URL}. "
            "Start LM Studio with its local server enabled and load the configured model."
        ) from exc
    except requests.exceptions.RequestException as exc:
        raise PromptExpansionError(
            f"LM Studio request failed at {LM_STUDIO_URL}: {exc}. "
            "Check the endpoint and local server settings."
        ) from exc

    if response.status_code != 200:
        detail = response.text.strip()
        raise PromptExpansionError(
            f"LM Studio returned HTTP {response.status_code} from {LM_STUDIO_URL}. "
            f"Check the loaded model and request schema. Response: {detail[:400]}"
        )

    try:
        result = response.json()
    except ValueError as exc:
        raise PromptExpansionError(
            "LM Studio returned invalid JSON; check that the chat-completions endpoint "
            "is configured and serving an OpenAI-compatible response."
        ) from exc

    choices = result.get("choices") if isinstance(result, dict) else None
    first_choice = choices[0] if isinstance(choices, list) and choices else None
    message = first_choice.get("message") if isinstance(first_choice, dict) else None
    content = message.get("content") if isinstance(message, dict) else None
    if not isinstance(content, str):
        raise PromptExpansionError(
            "LM Studio returned an invalid response: choices[0].message.content is missing "
            "or is not text. Verify the selected chat model and endpoint."
        )

    return _normalize_lm_content(content)


def expand_prompt_with_llm(
    user_intent: str,
    temperature: float = 0.7,
    max_tokens: int = 200,
) -> str:
    """Expand one user intent through LM Studio, or raise an actionable error."""
    if not isinstance(user_intent, str) or not user_intent.strip():
        raise ValueError("user_intent must be a non-empty string")

    system_prompt = """You are an Art Director assistant. Turn a short creative instruction into one concise, production-ready prompt for Stable Diffusion 1.5 and SD-Turbo. Include the subject, style, lighting, composition, mood, and useful quality details. Return only the prompt text, without commentary or quotation marks."""
    user_prompt = f"Transform this creative instruction into a detailed visual prompt: {user_intent.strip()}"
    return _request_lm_studio(
        system_prompt,
        user_prompt,
        temperature=temperature,
        max_tokens=max_tokens,
    )

## Step 4: Stable Diffusion 1.5 and SD-Turbo

**💡 Diffusion in one idea.** A diffusion model starts from pure random noise and repeatedly removes a little of it, each step nudging the image toward the prompt, until a picture emerges. More steps give more chances to refine detail.

- **SD15** uses the local `Art_Director/model/sd15` snapshot, 20 denoising steps, guidance 7.5, and accepts a negative prompt. Multi-step generation trades time for detail.
- **SD-Turbo** uses `Art_Director/model/sd-turbo`, exactly one step, guidance 0.0, and an empty negative prompt. It is *distilled* to produce an image in a single step, trading some control for speed.

**💡 Guidance (CFG).** `guidance_scale` controls how strongly the image obeys the prompt. Moderate values (SD15 uses 7.5) balance fidelity and quality; very high values over-saturate and distort. SD-Turbo is trained to skip guidance, so it uses 0.0—raising it there hurts rather than helps.

**💡 Memory.** These run in FP16. For the course setting (512×512) plan for roughly **16 GB system RAM** and **4 GB GPU/shared memory** at minimum (**32 GB / 6 GB** comfortable). A proxy measurement showed about **3.1 GiB** peak for SD15 and **2.94 GiB** for SD-Turbo—approximate guidance, not a promise for your hardware. On an APU the GPU shares system memory, and a loaded LM Studio model adds pressure. The runtime also enables attention and VAE slicing so activations are computed in chunks, lowering the peak further at a small speed cost while keeping the FP16 weights unchanged.

The notebook prints measured elapsed generation time; hardware, first-load cost, and memory pressure determine the value. No fixed timing claim is made.

**📄 License and safety.** The model assets and generated images remain subject to their model-card licenses and acceptable-use terms. SD15 identifies CreativeML OpenRAIL-M; for SD-Turbo, follow the pinned model repository LICENSE.md, model card, and current Stability AI commercial terms at https://huggingface.co/stabilityai/sd-turbo and https://stability.ai/license. SD-Turbo's configuration does not include a safety checker, and the DiffusionRuntime ROCm path does not disable one in code. Review outputs before use.

## Step 5: Prompt and image helpers

`ArtDirector.create_prompt` handles the user-facing expansion, `create_image` calls an already-owned runtime and reports measured elapsed time, and `refine_image` expands a refinement request before creating and closing one SD15 runtime. Generation errors propagate to the notebook so failures are visible.

**🔍 Observe.** This cell defines helpers only. The printed line confirms they are ready and that no diffusion model has been loaded yet.

In [ ]:
class ArtDirector:
    """Prompt workflow helpers; each caller owns its DiffusionRuntime."""

    def create_prompt(
        self,
        user_intent: str,
        use_llm_expansion: bool = True,
    ) -> str:
        if not use_llm_expansion:
            if not isinstance(user_intent, str) or not user_intent.strip():
                raise ValueError("user_intent must be a non-empty string")
            return user_intent.strip()

        print(f"User intent: {user_intent}")
        print("Expanding the prompt with LM Studio...")
        expanded_prompt = expand_prompt_with_llm(user_intent)
        print(f"Expanded prompt: {expanded_prompt}")
        return expanded_prompt

    def create_image(
        self,
        runtime: DiffusionRuntime,
        prompt: str,
        *,
        negative_prompt: str = "",
        height: int = 512,
        width: int = 512,
        steps: int | None = None,
        guidance_scale: float | None = None,
        count: int = 1,
        seed: int | None = None,
    ) -> tuple[list, float]:
        """Generate with an already-owned runtime and report measured elapsed time."""
        started = perf_counter()
        images = runtime.generate(
            prompt,
            negative_prompt=negative_prompt,
            height=height,
            width=width,
            steps=steps,
            guidance_scale=guidance_scale,
            count=count,
            seed=seed,
        )
        elapsed_seconds = perf_counter() - started
        print(
            f"{runtime.spec.name}: generated {len(images)} image(s) "
            f"in {elapsed_seconds:.3f} seconds"
        )
        return images, elapsed_seconds

    def refine_prompt(self, refinement_request: str, base_prompt: str) -> str:
        if not isinstance(refinement_request, str) or not refinement_request.strip():
            raise ValueError("refinement_request must be a non-empty string")
        if not isinstance(base_prompt, str) or not base_prompt.strip():
            raise ValueError("base_prompt must be a non-empty string")

        refinement_system_prompt = """You are an Art Director assistant. Rewrite the existing image prompt to apply the user's refinement request. Preserve unrelated subject, style, composition, mood, and quality details. Return only the complete revised prompt, without commentary or quotation marks."""
        user_prompt = f"""Existing prompt: {base_prompt.strip()}

Refinement request: {refinement_request.strip()}

Return a revised prompt that clearly applies the request."""
        refined_prompt = _request_lm_studio(
            refinement_system_prompt,
            user_prompt,
            temperature=0.7,
            max_tokens=200,
        )
        if refined_prompt.casefold() == base_prompt.strip().casefold():
            raise PromptExpansionError(
                "LM Studio returned the original prompt unchanged; make the refinement request more specific."
            )
        return refined_prompt

    def refine_image(
        self,
        refinement_request: str,
        base_prompt: str,
        *,
        model_name: str = "sd15",
        expected_gfx: str | None = None,
        negative_prompt: str = "",
        height: int = 512,
        width: int = 512,
        steps: int | None = None,
        guidance_scale: float | None = None,
        count: int = 1,
        seed: int | None = None,
    ) -> tuple[str, list, float, bool]:
        """Refine first, then own and close one runtime for the revised image."""
        refined_prompt = self.refine_prompt(refinement_request, base_prompt)
        runtime = None
        images: list = []
        elapsed_seconds = 0.0
        runtime_closed = False
        try:
            runtime = DiffusionRuntime(MODEL_SPECS[model_name], expected_gfx=expected_gfx)
            images, elapsed_seconds = self.create_image(
                runtime,
                refined_prompt,
                negative_prompt=negative_prompt,
                height=height,
                width=width,
                steps=steps,
                guidance_scale=guidance_scale,
                count=count,
                seed=seed,
            )
        finally:
            if runtime is not None:
                runtime.close()
                runtime_closed = runtime.closed
                del runtime
        return refined_prompt, images, elapsed_seconds, runtime_closed


art_director = ArtDirector()
print("ArtDirector helpers are ready; no diffusion model has been loaded.")

## Step 6: Expand one prompt with LM Studio

This step intentionally creates no DiffusionRuntime. If LM Studio is not reachable, fix the service configuration and rerun this cell before loading model weights.

**🔍 Observe.** Read the expanded prompt the LLM produced. Notice the added subject, style, lighting, composition, and mood.

**🧪 Try it.** Change `user_intent` to your own short idea and rerun to see how the expansion changes.

In [ ]:
print("=" * 70)
print("Step 6: Expand One Prompt (LM Studio only)")
print("=" * 70)

user_intent = "I want to draw a blue paradise bird in the jungle."
expanded_prompt = art_director.create_prompt(user_intent=user_intent)
assert expanded_prompt.strip(), "LM Studio must return a non-empty expanded prompt."
print("\nThis one expanded prompt is reused unchanged in Steps 7 and 8.")

## Step 7: Generate with SD15

The same expanded prompt is used at 512x512 with 20 steps, guidance 7.5, and seed 42. The image is displayed and saved. The `finally` block closes and deletes the runtime even if generation or saving fails.

**🔍 Observe.** Watch the step progress and the measured elapsed time. This is the multi-step baseline.

**🧪 Try it.** After you have compared both models, come back and vary `steps` (for example 10 vs. 30) or `guidance` to see their effect.

In [ ]:
print("=" * 70)
print("Step 7: SD15, 512x512, 20 steps, guidance 7.5")
print("=" * 70)

sd15_runtime = None
sd15_runtime_closed = False
try:
    sd15_runtime = DiffusionRuntime(MODEL_SPECS["sd15"], expected_gfx=EXPECTED_GFX)
    images_sd15, elapsed_sd15 = art_director.create_image(
        sd15_runtime,
        expanded_prompt,
        height=512,
        width=512,
        steps=20,
        guidance_scale=7.5,
        count=1,
        seed=42,
    )
    if len(images_sd15) != 1:
        raise RuntimeError(f"Expected one SD15 image, received {len(images_sd15)}")

    output_dir = project_root / "generated_images"
    output_dir.mkdir(parents=True, exist_ok=True)
    image_sd15 = images_sd15[0]
    display(image_sd15)
    image_path_sd15 = output_dir / "sd15_bird.png"
    image_sd15.save(image_path_sd15)
    print(f"Saved SD15 image to {image_path_sd15}")
finally:
    if sd15_runtime is not None:
        sd15_runtime.close()
        sd15_runtime_closed = sd15_runtime.closed
        del sd15_runtime

assert sd15_runtime_closed, "SD15 runtime must be closed before Step 8."
print(f"Measured SD15 generation time: {elapsed_sd15:.3f} seconds")

## Step 8: Generate with SD-Turbo

The assertion proves SD15 has been closed before SD-Turbo is constructed. The same prompt and seed are used at 512x512 with one step, guidance 0.0, and no negative prompt. The measured elapsed value is printed after cleanup.

**💡 What a seed does.** The seed fixes the random starting noise so a run is repeatable on the same setup. It does not guarantee pixel-identical output across different hardware or library versions—treat it as a fair, reproducible starting point, not a cross-device checksum.

**🔍 Observe.** Compare this one-step image with the SD15 image from the same prompt and seed: SD-Turbo is much faster, while SD15 usually holds more fine detail.

In [ ]:
print("=" * 70)
print("Step 8: SD-Turbo, 512x512, 1 step, guidance 0.0")
print("=" * 70)

assert sd15_runtime_closed, "SD15 must be closed before SD-Turbo is loaded."
sd_turbo_runtime = None
sd_turbo_runtime_closed = False
try:
    sd_turbo_runtime = DiffusionRuntime(
        MODEL_SPECS["sd-turbo"],
        expected_gfx=EXPECTED_GFX,
    )
    images_turbo, elapsed_turbo = art_director.create_image(
        sd_turbo_runtime,
        expanded_prompt,
        negative_prompt="",
        height=512,
        width=512,
        steps=1,
        guidance_scale=0.0,
        count=1,
        seed=42,
    )
    if len(images_turbo) != 1:
        raise RuntimeError(f"Expected one SD-Turbo image, received {len(images_turbo)}")

    output_dir = project_root / "generated_images"
    output_dir.mkdir(parents=True, exist_ok=True)
    image_turbo = images_turbo[0]
    display(image_turbo)
    image_path_turbo = output_dir / "sd-turbo_bird.png"
    image_turbo.save(image_path_turbo)
    print(f"Saved SD-Turbo image to {image_path_turbo}")
finally:
    if sd_turbo_runtime is not None:
        sd_turbo_runtime.close()
        sd_turbo_runtime_closed = sd_turbo_runtime.closed
        del sd_turbo_runtime

assert sd_turbo_runtime_closed, "SD-Turbo runtime must be closed after generation."
print(f"Measured SD-Turbo generation time: {elapsed_turbo:.3f} seconds")

## Step 9: Optional refinement

A refinement request goes through LM Studio, then one SD15 runtime is used and closed by `refine_image`. There is never a simultaneous SD15 and SD-Turbo runtime. If the external service is absent, the method raises rather than using the base prompt as a fallback.

**🧪 Try it.** Describe a change ("make it warmer, add fog") and refine. Because the previous runtime is already closed, only one model holds GPU memory at a time.

In [ ]:
print("=" * 70)
print("Step 9: Optional Refinement (one model at a time)")
print("=" * 70)

assert sd_turbo_runtime_closed, "The previous runtime must be closed before refinement."
refinement_request = "Make the bird red and use warm sunset lighting."
refined_prompt, refined_images, elapsed_refined, refinement_runtime_closed = art_director.refine_image(
    refinement_request=refinement_request,
    base_prompt=expanded_prompt,
    model_name="sd15",
    expected_gfx=EXPECTED_GFX,
    height=512,
    width=512,
    steps=20,
    guidance_scale=7.5,
    count=1,
    seed=42,
)
assert refinement_runtime_closed, "The refinement runtime must be closed by refine_image."
if len(refined_images) != 1:
    raise RuntimeError(f"Expected one refined image, received {len(refined_images)}")

output_dir = project_root / "generated_images"
output_dir.mkdir(parents=True, exist_ok=True)
display(refined_images[0])
image_path_refined = output_dir / "sd15_refined_bird.png"
refined_images[0].save(image_path_refined)
print(f"Refined prompt: {refined_prompt}")
print(f"Saved refined image to {image_path_refined}")
print(f"Measured refinement generation time: {elapsed_refined:.3f} seconds")

## Summary

1. State an intention and expand it once with LM Studio.
2. Compare the same prompt and seed with SD15's multi-step generation and SD-Turbo's one-step generation.
3. Observe the actual measured times and save the images.
4. Refine only after the prior runtime has been closed.

**✅ What you learned.** You turned a short intention into a directed visual prompt, saw how diffusion builds an image from noise, and compared a multi-step model with a distilled one-step model under a fair prompt-and-seed setup.

**⚠️ Model limitations.** Diffusion models struggle with legible text, correct hands and faces, and can reflect biases in their training data; they have no built-in understanding of truth or safety. Review generated images before sharing and apply content controls appropriate to your use.

**🧪 Extend.** Try different intention styles, vary `steps`/`guidance`/`seed`, compare aspect ratios, and study how negative prompts change SD15. The Docker setup in the project README selects architecture-specific wheels matching the active device; consult AMD's [official ROCm compatibility matrix](https://rocm.docs.amd.com/projects/radeon-ryzen/en/latest/docs/compatibility/compatibility.html). Run `python tests/smoke_art_director.py --output-dir <output-directory>` from the repository root to observe local sequential model behavior, output integrity, and memory cleanup; exercise LM Studio separately with a real external service. Use `AUP_EXPECTED_GFX` only for advanced troubleshooting when a diagnostic needs an explicit reported architecture.